In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


**Text Classification using Machine Learning Models**

In [10]:
import pandas as pd
import re
import string

import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report

nltk.download('punkt')
nltk.download('stopwords')

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

**Load Dataset**

In [11]:
data = pd.read_csv(
    r"/content/drive/MyDrive/AI and ML/trum_tweet_sentiment_analysis.csv",
    encoding="ISO-8859-1"
)

data.head()

,text,Sentiment
0,RT @JohnLeguizamo: #trump not draining swamp b...,0
1,ICYMI: Hackers Rig FM Radio Stations To Play A...,0
2,Trump protests: LGBTQ rally in New York https:...,1
3,"""Hi I'm Piers Morgan. David Beckham is awful b...",0
4,RT @GlennFranco68: Tech Firm Suing BuzzFeed fo...,0


In [12]:
print(data.columns)

Index(['text', 'Sentiment'], dtype='object')


**Text Cleaning and Tokenization**

In [15]:
stop_words = set(stopwords.words('english'))

def clean_text(text):
    text = text.lower()

    text = re.sub(r'http\S+|www\S+', '', text)
    text = re.sub(r'@\w+', '', text)
    text = re.sub(r'[^a-z\s]', '', text)

    # SIMPLE TOKENIZATION (no NLTK needed)
    tokens = text.split()

    tokens = [word for word in tokens if word not in stop_words]

    return " ".join(tokens)

In [16]:
data['clean_text'] = data['text'].astype(str).apply(clean_text)

data[['text', 'clean_text']].head()

,text,clean_text
0,RT @JohnLeguizamo: #trump not draining swamp b...,rt trump draining swamp taxpayer dollars trips...
1,ICYMI: Hackers Rig FM Radio Stations To Play A...,icymi hackers rig fm radio stations play antit...
2,Trump protests: LGBTQ rally in New York https:...,trump protests lgbtq rally new york bbcworld via
3,"""Hi I'm Piers Morgan. David Beckham is awful b...",hi im piers morgan david beckham awful donald ...
4,RT @GlennFranco68: Tech Firm Suing BuzzFeed fo...,rt tech firm suing buzzfeed publishing unverif...


**Train-Test Split**

In [18]:
X = data['clean_text']
y = data['Sentiment']

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

**TF-IDF Vectorization**

In [19]:
tfidf = TfidfVectorizer(max_features=5000)

X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf = tfidf.transform(X_test)

**Model Training and Evaluation**

In [20]:
model = LogisticRegression(max_iter=1000)
model.fit(X_train_tfidf, y_train)

LogisticRegression(max_iter=1000)

In [21]:
y_pred = model.predict(X_test_tfidf)

In [22]:
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.94      0.96      0.95    248842
           1       0.91      0.87      0.89    121183

    accuracy                           0.93    370025
   macro avg       0.92      0.92      0.92    370025
weighted avg       0.93      0.93      0.93    370025

